# 01 - Data Loading & Schema Profiling

**Goal**: Load all raw CSV tables, profile every column (dtype, null rate, unique count, samples),
and document the complete data dictionary.

**Learning concepts**: Pandas data loading, dtype specification, memory management for large CSVs,
understanding relational data schemas.

---

In [2]:
print(f"Importing libraries...")

import pandas as pd
import numpy as np
from talentlens.config import (
    POSTINGS_CSV, COMPANIES_CSV, COMPANY_INDUSTRIES_CSV,
    COMPANY_SPECIALITIES_CSV, EMPLOYEE_COUNTS_CSV,
    BENEFITS_CSV, JOB_INDUSTRIES_CSV, JOB_SKILLS_CSV,
    SALARIES_CSV, INDUSTRIES_MAP_CSV, SKILLS_MAP_CSV,
)
from talentlens.dataset import load_raw_postings, load_secondary_tables

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print(f"[SUCCESS] Libraries imported successfully.")

Importing libraries...
[SUCCESS] Libraries imported successfully.


## 1. Load the Main Postings Table

The main dataset has **3.38M rows** and is ~493MB. We load it with specified dtypes
to avoid pandas guessing wrong types (which wastes memory).

> **Tip**: For development, use `nrows=10000` to load a small sample first.
> Once your code works, remove the limit to process the full dataset.

In [3]:
# Load a small sample first to understand the schema
# Change nrows=None to load the full dataset when ready
df_postings = load_raw_postings()
print(f"Shape: {df_postings.shape}")
print(f"Memory usage: {df_postings.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df_postings.info()
df_postings.head()

2026-03-22 21:35:29.202 | INFO     | talentlens.dataset:load_raw_postings:50 - Loading raw postings from C:\Users\mark2\OneDrive\Documents\projects\TalentLens\data\raw\postings.csv
2026-03-22 21:35:33.216 | INFO     | talentlens.dataset:load_raw_postings:82 - Loaded 123,849 raw postings (31 columns)


Shape: (123849, 31)
Memory usage: 978.5 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     2

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing C...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,17.0,Full-time,2.0,1.713398e+12,NaN,https://www.linkedin.com/jobs/view/921716/?trk=jobs_biz_prem_srch,NaN,ComplexOnsiteApply,1.715990e+12,NaN,NaN,Requirements: \n\nWe are seeking a College or Graduate Student (can also be completed with schoo...,1.713398e+12,NaN,0.0,FULL_TIME,USD,BASE_SALARY,38480.0,08540,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committed to serving clients with best practices to help ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,30.0,Full-time,NaN,1.712858e+12,NaN,https://www.linkedin.com/jobs/view/1829192/?trk=jobs_biz_prem_srch,NaN,ComplexOnsiteApply,1.715450e+12,NaN,NaN,NaN,1.712858e+12,NaN,0.0,FULL_TIME,USD,BASE_SALARY,83200.0,80521,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting applications for an Assistant Restaurant Manager.\nWe offer h...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,45000.0,Full-time,NaN,1.713278e+12,NaN,https://www.linkedin.com/jobs/view/10998357/?trk=jobs_biz_prem_srch,NaN,ComplexOnsiteApply,1.715870e+12,NaN,NaN,We are currently accepting resumes for FOH - Asisstant Restaurant Management with a strong focus...,1.713278e+12,NaN,0.0,FULL_TIME,USD,BASE_SALARY,55000.0,45202,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associate Attorney,Senior Associate Attorney - Elder Law / Trusts and Estates Our legal team is committed to provi...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,140000.0,Full-time,NaN,1.712896e+12,NaN,https://www.linkedin.com/jobs/view/23221523/?trk=jobs_biz_prem_srch,NaN,ComplexOnsiteApply,1.715488e+12,NaN,NaN,This position requires a baseline understanding of online marketing including Search Engine Mark...,1.712896e+12,NaN,0.0,FULL_TIME,USD,BASE_SALARY,157500.0,11040,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience in commerical and industrial equipment. Minimum 5 ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,60000.0,Full-time,NaN,1.713452e+12,NaN,https://www.linkedin.com/jobs/view/35982263/?trk=jobs_biz_prem_srch,NaN,ComplexOnsiteApply,1.716044e+12,NaN,NaN,NaN,1.713452e+12,NaN,0.0,FULL_TIME,USD,BASE_SALARY,70000.0,52601,19057.0


## 2. Profile the Postings Columns

For each column, we want to know:
- **dtype**: What type did pandas assign?
- **null rate**: What percentage of values are missing?
- **unique count**: How many distinct values?
- **sample values**: What does the data actually look like?

In [4]:
def profile_dataframe(df: pd.DataFrame, name: str = "DataFrame") -> pd.DataFrame:
    """Generate a profiling summary for every column in a DataFrame."""
    profile = pd.DataFrame({
        'dtype': df.dtypes,
        'non_null': df.count(),
        'null_count': df.isna().sum(),
        'null_pct': (df.isna().sum() / len(df) * 100).round(1),
        'unique': df.nunique(),
        'sample_values': [df[col].dropna().head(3).tolist() for col in df.columns],
    })
    print(f"\n{'='*60}")
    print(f"  {name}: {len(df):,} rows x {len(df.columns)} columns")
    print(f"{'='*60}")
    return profile

In [5]:
profile_postings = profile_dataframe(df_postings, "Postings")
profile_postings


  Postings: 123,849 rows x 31 columns


,dtype,non_null,null_count,null_pct,unique,sample_values
job_id,int64,123849,0,0.0,123849,"[921716, 1829192, 10998357]"
company_name,object,122130,1719,1.4,24428,"[Corcoran Sawyer Smith, The National Exemplar , Abrams Fensterman, LLP]"
title,object,123849,0,0.0,72521,"[Marketing Coordinator, Mental Health Therapist/Counselor, Assitant Restaurant Manager]"
description,object,123842,7,0.0,107827,[Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing ...
max_salary,float64,29793,94056,75.9,5321,"[20.0, 50.0, 65000.0]"
pay_period,object,36073,87776,70.9,5,"[HOURLY, HOURLY, YEARLY]"
location,object,123849,0,0.0,8526,"[Princeton, NJ, Fort Collins, CO, Cincinnati, OH]"
company_id,float64,122132,1717,1.4,24474,"[2774458.0, 64896719.0, 766262.0]"
views,float64,122160,1689,1.4,684,"[20.0, 1.0, 8.0]"
med_salary,float64,6280,117569,94.9,1417,"[350.0, 25.0, 23.0]"


## 3. Load & Profile All Secondary Tables

The dataset is **relational** — like a database with multiple linked tables:

```
postings (main) ──┬── job_skills ──── skills (mapping)
                   ├── job_industries ── industries (mapping)
                   ├── benefits
                   ├── salaries
                   └── companies ──┬── company_industries
                                   ├── company_specialities
                                   └── employee_counts
```

Tables are linked via `job_id` or `company_id` foreign keys.

In [6]:
tables = load_secondary_tables()

for name, df in tables.items():
    display(profile_dataframe(df, name))

2026-03-22 21:36:07.920 | INFO     | talentlens.dataset:load_secondary_tables:114 - Loaded companies: 24,473 rows
2026-03-22 21:36:07.938 | INFO     | talentlens.dataset:load_secondary_tables:114 - Loaded company_industries: 24,375 rows
2026-03-22 21:36:07.999 | INFO     | talentlens.dataset:load_secondary_tables:114 - Loaded company_specialities: 169,387 rows
2026-03-22 21:36:08.018 | INFO     | talentlens.dataset:load_secondary_tables:114 - Loaded employee_counts: 35,787 rows
2026-03-22 21:36:08.043 | INFO     | talentlens.dataset:load_secondary_tables:114 - Loaded benefits: 67,943 rows
2026-03-22 21:36:08.083 | INFO     | talentlens.dataset:load_secondary_tables:114 - Loaded job_industries: 164,808 rows
2026-03-22 21:36:08.120 | INFO     | talentlens.dataset:load_secondary_tables:114 - Loaded job_skills: 213,768 rows
2026-03-22 21:36:08.146 | INFO     | talentlens.dataset:load_secondary_tables:114 - Loaded salaries: 40,785 rows
2026-03-22 21:36:08.156 | INFO     | talentlens.dataset


  companies: 24,473 rows x 10 columns


,dtype,non_null,null_count,null_pct,unique,sample_values
company_id,int64,24473,0,0.0,24473,"[1009, 1016, 1025]"
name,object,24472,1,0.0,24428,"[IBM, GE HealthCare, Hewlett Packard Enterprise]"
description,object,24176,297,1.2,24164,"[At IBM, we do more than work. We create. We create as technologists, developers, and engineers...."
company_size,float64,21699,2774,11.3,7,"[7.0, 7.0, 7.0]"
state,object,24451,22,0.1,788,"[NY, 0, Texas]"
country,object,24473,0,0.0,81,"[US, US, US]"
city,object,24472,1,0.0,4124,"[Armonk, New York, Chicago, Houston]"
zip_code,object,24445,28,0.1,7779,"[10504, 0, 77389]"
address,object,24451,22,0.1,19476,"[International Business Machines Corp., -, 1701 E Mossy Oaks Rd Spring]"
url,object,24473,0,0.0,24473,"[https://www.linkedin.com/company/ibm, https://www.linkedin.com/company/gehealthcare, https://ww..."



  company_industries: 24,375 rows x 2 columns


,dtype,non_null,null_count,null_pct,unique,sample_values
company_id,int64,24375,0,0.0,24365,"[391906, 22292832, 20300]"
industry,object,24375,0,0.0,144,"[Book and Periodical Publishing, Construction, Banking]"


...

,dtype,non_null,null_count,null_pct,unique,sample_values
salary_id,int64,40785,0,0.0,40785,"[1, 2, 3]"
job_id,int64,40785,0,0.0,40785,"[3884428798, 3887470552, 3884431523]"
max_salary,float64,33947,6838,16.8,5321,"[25.0, 120000.0, 200000.0]"
med_salary,float64,6838,33947,83.2,1417,"[20.0, 48.43, 105000.0]"
min_salary,float64,33947,6838,16.8,4620,"[23.0, 100000.0, 10000.0]"
pay_period,object,40785,0,0.0,5,"[HOURLY, HOURLY, YEARLY]"
currency,object,40785,0,0.0,6,"[USD, USD, USD]"
compensation_type,object,40785,0,0.0,1,"[BASE_SALARY, BASE_SALARY, BASE_SALARY]"



  industries_map: 422 rows x 2 columns


,dtype,non_null,null_count,null_pct,unique,sample_values
industry_id,int64,422,0,0.0,422,"[1, 3, 4]"
industry_name,object,388,34,8.1,388,"[Defense and Space Manufacturing, Computer Hardware Manufacturing, Software Development]"



  skills_map: 35 rows x 2 columns


,dtype,non_null,null_count,null_pct,unique,sample_values
skill_abr,object,35,0,0.0,35,"[ART, DSGN, ADVR]"
skill_name,object,35,0,0.0,35,"[Art/Creative, Design, Advertising]"


## 4. Explore the Mapping Tables

These are small lookup tables that decode IDs into human-readable names.

In [7]:
# Skills mapping: 35 skill categories
print("=== Skills Mapping ===")
display(tables['skills_map'])

print(f"\n=== Industries Mapping (first 20 of {len(tables['industries_map'])}) ===")
display(tables['industries_map'].head(20))

=== Skills Mapping ===


,skill_abr,skill_name
0,ART,Art/Creative
1,DSGN,Design
2,ADVR,Advertising
3,PRDM,Product Management
4,DIST,Distribution
5,EDU,Education
6,TRNG,Training
7,PRJM,Project Management
8,CNSL,Consulting
9,PRCH,Purchasing



=== Industries Mapping (first 20 of 422) ===


,industry_id,industry_name
0,1,Defense and Space Manufacturing
1,3,Computer Hardware Manufacturing
2,4,Software Development
3,5,Computer Networking Products
4,6,"Technology, Information and Internet"
5,7,Semiconductor Manufacturing
6,8,Telecommunications
7,9,Law Practice
8,10,Legal Services
9,11,Business Consulting and Services


## 5. Join job_skills with Skills Names

The `job_skills` table uses abbreviations like `IT`, `SALE`, `DSGN`.
Let's join with the mapping table to see the full skill names.

In [8]:
# Join skills abbreviations to full names
skills_enriched = tables['job_skills'].merge(
    tables['skills_map'],
    on='skill_abr',
    how='left'
)
print(f"Job-skill associations: {len(skills_enriched):,}")
print(f"\nTop 15 skills by frequency:")
skills_enriched['skill_name'].value_counts().head(15)

Job-skill associations: 213,768

Top 15 skills by frequency:


skill_name
Information Technology    26137
Sales                     22475
Management                20861
Manufacturing             18185
Health Care Provider      17369
Business Development      14290
Engineering               13009
Other                     12608
Finance                    8540
Marketing                  5525
Accounting/Auditing        5461
Administrative             4860
Customer Service           4292
Project Management         3997
Analyst                    3858
Name: count, dtype: int64

## 6. Quick Stats Summary

Before moving to cleaning, let's capture the key numbers.

In [9]:
# Get total line count using Python (cross-platform, no wc needed)
import os
csv_size_mb = os.path.getsize(POSTINGS_CSV) / 1e6
# We already know from the sample how many columns there are;
# for total rows, use the loaded sample or note the known count.
total_lines = "~3,383,602 (from CSV header count)"

print("=== Dataset Summary ===")
print(f"Raw CSV file size: {csv_size_mb:.0f} MB")
print(f"Total postings (raw CSV lines): {total_lines}")
print(f"Sample loaded: {len(df_postings):,} rows")
print(f"Columns: {df_postings.shape[1]}")
print(f"\nSecondary tables:")
for name, df in tables.items():
    print(f"  {name}: {len(df):,} rows x {df.shape[1]} cols")

print(f"\nKey null rates in postings (sample):")
key_cols = ['description', 'title', 'med_salary', 'remote_allowed', 
            'formatted_experience_level', 'skills_desc', 'applies', 'views']
for col in key_cols:
    if col in df_postings.columns:
        null_pct = df_postings[col].isna().mean() * 100
        print(f"  {col}: {null_pct:.1f}% null")

=== Dataset Summary ===
Raw CSV file size: 517 MB
Total postings (raw CSV lines): ~3,383,602 (from CSV header count)
Sample loaded: 123,849 rows
Columns: 31

Secondary tables:
  companies: 24,473 rows x 10 cols
  company_industries: 24,375 rows x 2 cols
  company_specialities: 169,387 rows x 2 cols
  employee_counts: 35,787 rows x 4 cols
  benefits: 67,943 rows x 3 cols
  job_industries: 164,808 rows x 2 cols
  job_skills: 213,768 rows x 2 cols
  salaries: 40,785 rows x 8 cols
  industries_map: 422 rows x 2 cols
  skills_map: 35 rows x 2 cols

Key null rates in postings (sample):
  description: 0.0% null
  title: 0.0% null
  med_salary: 94.9% null
  remote_allowed: 87.7% null
  formatted_experience_level: 23.7% null
  skills_desc: 98.0% null
  applies: 81.2% null
  views: 1.4% null


## Key Observations

*(Fill in after running the notebook)*

1. **Postings table**: X rows, Y columns. Description null rate = Z%.
2. **Salary coverage**: Only X% of postings have salary data.
3. **Skills coverage**: X job-skill associations across Y unique skills.
4. **Next step**: Clean the data in notebook 02.

---

**→ Next notebook**: `02-mp-data-cleaning.ipynb`